In [ ]:
import sagemaker
import boto3
from botocore.exceptions import ClientError
import ast
import datasets

AWS_ACCESS_KEY_ID=nAWS_SECRET_ACCESS_KEY=
AWS_SESSION_TOKEN=

session = sagemaker.Session()

sagemaker_session_bucket=None
if sagemaker_session_bucket is None and session is not None:
    # set to default bucket if a bucket name is not given
    sagemaker_session_bucket = session.default_bucket()
    
role = sagemaker.get_execution_role()

sess = sagemaker.Session(default_bucket=sagemaker_session_bucket)

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {sess.boto_region_name}")

In [ ]:
from sagemaker.huggingface import get_huggingface_llm_image_uri

# retrieve the llm image uri
llm_image = get_huggingface_llm_image_uri(
  "huggingface",
  version="1.0.3"
)

# print ecr image uri
print(f"llm image uri: {llm_image}")


In [ ]:
def get_secret_hf(session):

    secret_name = "hf-access-token"
    region_name = "ca-central-1"

    # Create a Secrets Manager client
    client = session.client(
        service_name='secretsmanager',
        region_name=region_name
    )
    
    try:
        get_secret_value_response = client.get_secret_value(
            SecretId=secret_name
        )
    except ClientError as e:
        # For a list of exceptions thrown, see
        # https://docs.aws.amazon.com/secretsmanager/latest/apireference/API_GetSecretValue.html
        raise e

    # Decrypts secret using the associated KMS key.
    secret = get_secret_value_response['SecretString']

    # Your code goes here.
    return ast.literal_eval(secret)['hf-access-token']

session = boto3.session.Session(
    aws_access_key_id=AWS_ACCESS_KEY_ID,
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY,
    aws_session_token=AWS_SESSION_TOKEN
)

hf_token = get_secret_hf(session)

In [ ]:
import json
from sagemaker.huggingface import HuggingFaceModel
from sagemaker.async_inference.async_inference_config import AsyncInferenceConfig

# sagemaker config
instance_type = "ml.g5.48xlarge"
number_of_gpu = 8
health_check_timeout = 600

# TGI config
config = {
  'HF_MODEL_ID': "meta-llama/Llama-2-70b-chat-hf", # model_id from hf.co/models
  'SM_NUM_GPUS': json.dumps(number_of_gpu), # Number of GPU used per replica
  'MAX_INPUT_LENGTH': json.dumps(3500),  # Max length of input text
  'MAX_TOTAL_TOKENS': json.dumps(4096),  # Max length of the generation (including input text)
  'MAX_BATCH_TOTAL_TOKENS': json.dumps(8192),  # Limits the number of tokens that can be processed in parallel during the generation
  'HF_MODEL_QUANTIZE': "bitsandbytes", 
  'HUGGING_FACE_HUB_TOKEN': hf_token
}

async_config = AsyncInferenceConfig(
    output_path= "s3://eko-ekoka-ai-project/data/generation_output/Feb5_generation_2" ,
)


# create HuggingFaceModel
llm_model = HuggingFaceModel(
  role=role,
  image_uri=llm_image,
  env=config
)

async_predictor = llm_model.deploy(
    initial_instance_count=1,
    instance_type=instance_type,
    async_inference_config=async_config,
    tags=[{"Key":'uw-ai-note-generation', "Value":'generation-evaluation'}],
    vpc_config_override = {"Subnets": ['subnet-0ba981d4c6314f9a9'], "SecurityGroupIds": ['sg-04645476ef00795e3'] },
    
)




In [ ]:
# Deploy model to an endpoint
# https://sagemaker.readthedocs.io/en/stable/api/inference/model.html#sagemaker.model.Model.deploy
# llm = llm_model.deploy(
#   initial_instance_count=1,
#   instance_type=instance_type,
#   # volume_size=400, # If using an instance with local SSD storage, volume_size must be None, e.g. p4 but not p3
#   container_startup_health_check_timeout=health_check_timeout, # 10 minutes to be able to load the model
#   tags=[{"Key":'uw-ai-note-generation', "Value":'generation-debugging'}],
#   vpc_config_override = {"Subnets": ['subnet-0ba981d4c6314f9a9'], "SecurityGroupIds": ['sg-04645476ef00795e3'] }
  
# )


In [ ]:
generated_vs_curated = pd.read_csv('data/Feb2_generation.csv')
generated_vs_curated = generated_vs_curated[generated_vs_curated['PN_id'] != 0]

In [ ]:
import pandas as pd
df = pd.read_csv('data/curated_examples.csv', encoding='MacRoman')
df = df.dropna(axis=1, how='all')

prefix_PN = "This is a Progress Note - OT from "
prefix_SN = "This is a Scratch Note - OT from "
df['PN'] = df.apply(lambda row: prefix_PN + row['Date'] + ":\n" + row['PN'], axis=1)
df['SN'] = df.apply(lambda row: prefix_SN + row['Date'] + ":\n" + row['SN'], axis=1)

In [ ]:
curated_examples_sample

In [ ]:
curated_examples_sample = df.sample(n=5, random_state=30)
curated_examples_sample = curated_examples_sample.reset_index()
curated_examples_sample = curated_examples_sample.drop(index=0)

In [ ]:
generated_vs_curated

In [ ]:
demos = curated_examples_sample.join(generated_vs_curated.set_index('PN_id'))
demos_dict_list = demos.to_dict(orient='records')

In [ ]:
demos

In [ ]:
def build_llama_prompt(message, system_prompt = 'default', id_string='00000'):
    start_prompt = "<s>[INST] "
    end_prompt = " [/INST]"
    if system_prompt == 'default':
        system_prompt = f"""<<SYS>>\nYou are a helpful, respectful, and honest assistant that follows these rules:
1. Follow directions meticulously
2. Do not produce any unnecessary extra text
3. Answer as helpfully as possible, while being safe.
The ID for this generation is {id_string}.\n<</SYS>>\n\n"""
    
    content = message
    return start_prompt + system_prompt + content + end_prompt

In [ ]:
# dataset = datasets.Dataset.load_from_disk('data/ft_dataset_redact')

# progress_notes_sample = dataset.train_test_split(test_size=5, seed=36)['test']

In [ ]:

prompt_list = []

for x in range(2):
   
    # for i in range(0, len(demos_dict_list), 2):
        
    message_examples = ""

    pair1 = demos_dict_list[2]
    if i + 1 < len(demos_dict_list):
        pair2 = demos_dict_list[3]

    message_examples = message_examples + f"""\n[User]: Here is a first draft of a Scratch Note. Make it more concise. Capture the main points in as few words as possible and exclude any statements about COVID-19 safety measures.
{pair1['generated_output']}
[System]: Here is a more concise version of the Scratch Note.
{pair1['SN']}
[User]: Here is a first draft of a Scratch Note. Make it more concise. Capture the main points in as few words as possible and exclude any statements about COVID-19 safety measures.
{pair2['generated_output']}
[System]: Here is a more concise version of the Scratch Note.
{pair2['SN']}"""


    message = message_examples + f"""\n[User]: Here is a first draft of a Scratch Note. Make it more concise. Capture the main points in as few words as possible and exclude any statements about COVID-19 safety measures.
{demos_dict_list[x]['generated_output']}
[System]: Here is a more concise version of the Scratch Note. """

    id_string = f"{demos_dict_list[x]['ID']}, {pair1['ID']}, {pair2['ID']}"
    print(id_string)
    prompt = build_llama_prompt(message = message, id_string= id_string)
    prompt_list.append(prompt)
    print("Curated examples used: " + str(pair1['ID']) + ", " + str(pair2['ID']))




In [ ]:
# progress_notes_list = progress_notes_sample['input_text']


# prompt_list = []

# for x in range(len(progress_notes_list)):
   
#     for i in range(0, len(demos_dict_list), 2):
        
#         message_examples = ""
        
#         pair1 = demos_dict_list[i]
#         if i + 1 < len(demos_dict_list):
#             pair2 = demos_dict_list[i + 1]
            
#         message_examples = message_examples + f"""\n[User]: Here is a Progress Note in SOAP format. Please write a matching Scratch Note.
# {pair1['PN']}
# [System]:Here is a Scratch Note that is based on the Progress Note.
# {pair1['SN']}
# [User]: Here is a Progress Note in SOAP format. Please write a matching Scratch Note.
# {pair2['PN']}
# [System]:Here is a Scratch Note that is based on the Progress Note.
# {pair2['SN']}"""
        
        
#         message = message_examples + f"""\n[User]: Here is a Progress Note in SOAP format. Please write a matching Scratch Note.
# {progress_notes_list[x]}
# [System]: Here is a Scratch Note that is based on the Progress Note. """
        
#         id_string = f"{x}, {pair1['ID']}, {pair2['ID']}"
#         print(id_string)
#         prompt = build_llama_prompt(message = message, id_string= id_string)
#         prompt_list.append(prompt)
#         print("Curated examples used: " + str(pair1['ID']) + ", " + str(pair2['ID']))




In [ ]:
prompt_list

In [ ]:
from transformers import LlamaTokenizer
llama_tokenizer = LlamaTokenizer.from_pretrained('meta-llama/Llama-2-70b', token=hf_token)

In [ ]:
    
for i in range(len(prompt_list)):  
    input_ids = llama_tokenizer(prompt_list[i])["input_ids"]
    # print(input_ids)
    num_input_tokens = len(input_ids)
    print("Number of tokens in the prompt: " + str(num_input_tokens))

In [ ]:
import time 

response_list = []

start = time.time()

for prompt in prompt_list:
    input_ids = llama_tokenizer(prompt)["input_ids"]
    # print(input_ids)
    num_input_tokens = len(input_ids)
    max_tokens = 4096 - num_input_tokens
    
    payload = {
  "inputs": prompt,
  "parameters": {
    "do_sample": True,
    "top_p": 0.9,
    "temperature": 0.8,
    "max_new_tokens": max_tokens,
    "repetition_penalty": 1.03,
    "stop": ["\nUser:","<|endoftext|>","</s>"]
    }
    }
    
    
    response = async_predictor.predict(payload)
    response_list.append(response)
    
print(f"Time taken: {time.time() - start}s")


In [ ]:
# # hyperparameters for llm
# payload = {
#   "inputs": prompt,
#   "parameters": {
#     "do_sample": True,
#     "top_p": 0.9,
#     "temperature": 0.8,
#     "max_new_tokens": 1024,
#     "repetition_penalty": 1.03,
#     "stop": ["\nUser:","<|endoftext|>","</s>"]
#   }
# }

# # send request to endpoint
# response = async_predictor.predict(payload)

# # print assistant respond
# # assistant = response[0]["generated_text"][len(prompt):]


In [ ]:
llm.delete_model()
llm.delete_endpoint()

In [ ]:
# !aws s3 cp s3://eko-ekoka-ai-project/data/generation_output/December18_ data/ --sse='AES256' --recursive

In [ ]:
import pandas as pd
df = pd.read_csv('data/curated_examples.csv', encoding='MacRoman')
df = df.dropna(axis=1, how='all')

prefix_PN = "This is a Progress Note - OT from "
prefix_SN = "This is a Scratch Note - OT from "
df['PN'] = df.apply(lambda row: prefix_PN + row['Date'] + ":\n" + row['PN'], axis=1)
df['SN'] = df.apply(lambda row: prefix_SN + row['Date'] + ":\n" + row['SN'], axis=1)

In [ ]:
import chardet
with open('data/curated_examples.csv', 'rb') as f:
    result = chardet.detect(f.read())
print(result)

In [ ]:
# demo_examples = [{"scratch_note": scratch_note_1, "progress_note": progress_note_1}]

In [ ]:
# progress_notes_list = progress_notes_sample['input_text']

# prompt_list = []
    
# for note in progress_notes_list:
    
#     message_examples = ""
#     for pair in demo_examples:
#         message_examples = message_examples + f"""\n[User]: Here is a Progress Note in SOAP format.
# {pair['progress_note']}
# [System]:Here is a Scratch Note that is based on the Progress Note.
# {pair['scratch_note']}"""
    
#         message = message_examples + f"""\n[User]: Here is a Progress Note in SOAP format.
# {note}
# [System]: Here is a Scratch Note that is based on the Progress Note. """
    
#     prompt = build_llama_prompt(message = message)
#     prompt_list.append(prompt)
    
# # print(prompt)